# Lynx Retrieval Visualization

This notebook loads a `lynx_report` JSON and visualizes a query sequence matched to its top-1 gallery sequence with keypoints and LightGlue matches.

In [ ]:
import json
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
from PIL import Image
import torch
import matplotlib.pyplot as plt
import matplotlib.cm as cm

import sys
repo_root = Path.cwd().resolve()
if repo_root.name == "scripts":
    repo_root = repo_root.parent
if not (repo_root / "README.md").exists():
    for parent in Path.cwd().resolve().parents:
        if (parent / "README.md").exists():
            repo_root = parent
            break
sys.path.append(str(repo_root))

from RDD.RDD import build as build_rdd
from RDD.matchers import LightGlue

In [ ]:
# Config
dataset_root = Path('/shared/sets/datasets/confidential/lynx/processed_frames/segmented/dfk-hr-cleaned')
report_path = repo_root / "outputs" / "lynx_report_2026_01_21-12_07_00.json"
cache_dir = repo_root / "outputs" / "lynx_cache"

rdd_weights = Path("./weights/RDD-v2.pth")
lg_weights = Path("./weights/RDD_lg-v2.pth")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

resize_max = 1024
top_k = 2048
top_m_pool = 5
matcher_threshold = 0.25

use_cached = True
show_failures = False
query_index = 5
max_matches = 20

In [ ]:
@dataclass
class FrameFeat:
    keypoints: np.ndarray
    descriptors: np.ndarray
    scores: np.ndarray
    image_size: np.ndarray

@dataclass
class SequenceEntry:
    split: str
    lynx_id: str
    site: str
    sequence_id: str
    frame_paths: List[Path]

    @property
    def name(self) -> str:
        return f"{self.lynx_id}/{self.site}/{self.sequence_id}"

def list_sequences(root: Path, split: str) -> List[SequenceEntry]:
    entries: List[SequenceEntry] = []
    split_dir = root / split
    for lynx_dir in sorted(split_dir.iterdir()):
        if not lynx_dir.is_dir():
            continue
        lynx_id = lynx_dir.name
        for site_dir in sorted(lynx_dir.iterdir()):
            if not site_dir.is_dir():
                continue
            site = site_dir.name
            for seq_dir in sorted(site_dir.iterdir()):
                if not seq_dir.is_dir():
                    continue
                frames = sorted(seq_dir.glob("frame_*.jpg"))
                if not frames:
                    continue
                entries.append(
                    SequenceEntry(
                        split=split,
                        lynx_id=lynx_id,
                        site=site,
                        sequence_id=seq_dir.name,
                        frame_paths=frames,
                    )
                )
    return entries

def sample_frames(frame_paths: List[Path], max_frames: int) -> List[Path]:
    if len(frame_paths) <= max_frames:
        return frame_paths
    idxs = np.linspace(0, len(frame_paths) - 1, num=max_frames, dtype=int)
    return [frame_paths[i] for i in idxs]

def load_image(path: Path, resize_max: int) -> torch.Tensor:
    img = Image.open(path).convert("RGB")
    if resize_max and resize_max > 0:
        w, h = img.size
        scale = resize_max / max(w, h)
        if scale < 1.0:
            img = img.resize((int(w * scale), int(h * scale)), Image.BILINEAR)
    arr = np.array(img).astype(np.float32) / 255.0
    tensor = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
    return tensor

def feat_cache_path(frame_path: Path) -> Path:
    rel = frame_path.relative_to(dataset_root)
    return cache_dir / rel.parent / f"{frame_path.stem}.npz"

def ensure_cache(path: Path, feat: FrameFeat):
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        path,
        keypoints=feat.keypoints,
        descriptors=feat.descriptors,
        scores=feat.scores,
        image_size=feat.image_size,
    )

def load_cached_feat(path: Path) -> FrameFeat:
    data = np.load(path)
    return FrameFeat(
        keypoints=data["keypoints"],
        descriptors=data["descriptors"],
        scores=data["scores"],
        image_size=data["image_size"],
    )

@torch.no_grad()
def extract_frame(model, img: torch.Tensor, device: torch.device, top_k: int) -> FrameFeat:
    model.top_k = top_k
    model.set_softdetect(top_k=top_k)
    img = img.to(device)
    out = model.extract(img)[0]
    return FrameFeat(
        keypoints=out["keypoints"].cpu().numpy(),
        descriptors=out["descriptors"].cpu().numpy(),
        scores=out["scores"].cpu().numpy(),
        image_size=np.array(img.shape[-2:], dtype=np.int32),
    )

def build_lightglue(device: torch.device):
    lg_conf = {
        "name": "lightglue",
        "input_dim": 256,
        "descriptor_dim": 256,
        "add_scale_ori": False,
        "n_layers": 9,
        "num_heads": 4,
        "flash": True,
        "mp": False,
        "filter_threshold": 0.01,
        "depth_confidence": -1,
        "width_confidence": -1,
        "weights": str(lg_weights),
    }
    return LightGlue("rdd", **lg_conf).to(device).eval()

def ensure_frame_feat(
    frame_path: Path,
    model,
    device: torch.device,
    use_cached: bool,
) -> FrameFeat:
    global rdd_model
    cache_path = feat_cache_path(frame_path)
    if use_cached and cache_path.exists():
        return load_cached_feat(cache_path)
    if model is None:
        model = build_rdd(None, weights=str(rdd_weights)).to(device).eval()
        rdd_model = model
    img = load_image(frame_path, resize_max)
    feat = extract_frame(model, img, device, top_k)
    ensure_cache(cache_path, feat)
    return feat

@torch.no_grad()
def match_frames(lg, fa: FrameFeat, fb: FrameFeat, device: torch.device):
    k0 = torch.from_numpy(fa.keypoints).to(device).unsqueeze(0)
    k1 = torch.from_numpy(fb.keypoints).to(device).unsqueeze(0)
    d0 = torch.from_numpy(fa.descriptors).to(device).unsqueeze(0)
    d1 = torch.from_numpy(fb.descriptors).to(device).unsqueeze(0)
    size0 = torch.tensor(fa.image_size[::-1].copy(), device=device).unsqueeze(0)
    size1 = torch.tensor(fb.image_size[::-1].copy(), device=device).unsqueeze(0)
    pred = lg(
        {
            "image0": {"keypoints": k0, "descriptors": d0, "image_size": size0},
            "image1": {"keypoints": k1, "descriptors": d1, "image_size": size1},
        }
    )
    matches = pred["matches"][0]
    conf = pred["scores"][0]
    if conf.numel() == 0:
        return np.zeros((0, 2)), np.zeros((0, 2)), np.zeros((0,))
    mkpts0 = fa.keypoints[matches[:, 0].cpu().numpy()]
    mkpts1 = fb.keypoints[matches[:, 1].cpu().numpy()]
    return mkpts0, mkpts1, conf.cpu().numpy()

def pair_score(lg, fa: FrameFeat, fb: FrameFeat, device: torch.device) -> float:
    mkpts0, mkpts1, conf = match_frames(lg, fa, fb, device)
    if len(conf) == 0:
        return 0.0
    norm = min(max(1, fa.keypoints.shape[0]), max(1, fb.keypoints.shape[0]))
    return float(conf.sum() / norm)

def find_best_pair(
    lg,
    q_frames: List[Path],
    g_frames: List[Path],
    rdd_model,
    device: torch.device,
    use_cached: bool,
) -> Tuple[Path, Path, FrameFeat, FrameFeat]:
    best_score = -1.0
    best_pair = (None, None, None, None)
    for qf in q_frames:
        fq = ensure_frame_feat(qf, rdd_model, device, use_cached)
        for gf in g_frames:
            fg = ensure_frame_feat(gf, rdd_model, device, use_cached)
            score = pair_score(lg, fq, fg, device)
            if score > best_score:
                best_score = score
                best_pair = (qf, gf, fq, fg)
    return best_pair

def draw_matches(
    img0: np.ndarray,
    img1: np.ndarray,
    kpts0: np.ndarray,
    kpts1: np.ndarray,
    conf: np.ndarray,
    max_matches: int = 200,
    title: str = "",
):
    if len(conf) > max_matches:
        order = np.argsort(-conf)[:max_matches]
        kpts0 = kpts0[order]
        kpts1 = kpts1[order]
        conf = conf[order]

    h0, w0 = img0.shape[:2]
    h1, w1 = img1.shape[:2]
    out = np.zeros((max(h0, h1), w0 + w1, 3), dtype=np.uint8)
    out[:h0, :w0] = img0
    out[:h1, w0:] = img1

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.imshow(out)
    ax.axis("off")
    if title:
        ax.set_title(title)

    if len(conf) == 0:
        return

    conf_norm = (conf - conf.min()) / (conf.max() - conf.min() + 1e-8)
    colors = cm.viridis(conf_norm)

    for (x0, y0), (x1, y1), c in zip(kpts0, kpts1, colors):
        ax.plot([x0, x1 + w0], [y0, y1], color=c, linewidth=1)

    ax.scatter(kpts0[:, 0], kpts0[:, 1], s=6, c=colors, marker="o")
    ax.scatter(kpts1[:, 0] + w0, kpts1[:, 1], s=6, c=colors, marker="o")
    plt.show()

In [ ]:
# Load report + sequence maps
with open(report_path, "r") as f:
    report = json.load(f)

results = report.get("results", [])

train_seqs = list_sequences(dataset_root, "train")
test_seqs = list_sequences(dataset_root, "test")
print(f"Train sequences: {len(train_seqs)}, Test sequences: {len(test_seqs)}")
train_map = {s.name: s for s in train_seqs}
test_map = {s.name: s for s in test_seqs}

if show_failures:
    filtered = [r for r in results if r.get("gt_id") != r.get("top1_id")]
else:
    filtered = [r for r in results if r.get("gt_id") == r.get("top1_id")]

if not filtered:
    raise RuntimeError("No results after filtering. Toggle show_failures or check report.")
print(f"Filtered results: {len(filtered)}")
query_rec = filtered[query_index % len(filtered)]
query_name = query_rec["query"]
top1_name = query_rec.get("top1")

q_seq = test_map.get(query_name)
g_seq = train_map.get(top1_name)

if q_seq is None or g_seq is None:
    raise RuntimeError("Could not resolve query or top1 sequence from report.")

print("Query:", q_seq.name)
print("Top-1:", g_seq.name)

In [ ]:
# Build models
lg_model = build_lightglue(device)
rdd_model = build_rdd(None, weights=str(rdd_weights))
rdd_model.to(device).eval()
if use_cached:
    cache_dir.mkdir(parents=True, exist_ok=True)

q_frames = sample_frames(q_seq.frame_paths, top_m_pool)
g_frames = sample_frames(g_seq.frame_paths, top_m_pool)

qf_path, gf_path, fq, fg = find_best_pair(
    lg_model,
    q_frames,
    g_frames,
    rdd_model,
    device,
    use_cached,
)

mkpts0, mkpts1, conf = match_frames(lg_model, fq, fg, device)
if matcher_threshold > 0:
    keep = conf >= matcher_threshold
    mkpts0 = mkpts0[keep]
    mkpts1 = mkpts1[keep]
    conf = conf[keep]

img0 = np.array(Image.open(qf_path).convert("RGB"))
img1 = np.array(Image.open(gf_path).convert("RGB"))
if resize_max and resize_max > 0:
    def resize_for_vis(img):
        h, w = img.shape[:2]
        scale = resize_max / max(h, w)
        if scale < 1.0:
            return np.array(Image.fromarray(img).resize((int(w * scale), int(h * scale)), Image.BILINEAR))
        return img
    img0 = resize_for_vis(img0)
    img1 = resize_for_vis(img1)

title = f"{q_seq.name} -> {g_seq.name} | matches: {len(conf)}"
draw_matches(img0, img1, mkpts0, mkpts1, conf, max_matches=max_matches, title=title)